# Продуктовая аналитика и A/B-тестирование в цифровом маркетплейсе
## Оценка результатов эксперимента с новой AI-моделью рекомендаций

**Команда:** К недрам Чили


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

## 0. Загрузка данных и первичный аудит представленного датасета

Датасет состоит из 10 таблиц: пользователи, распределение по группам эксперимента,
сессии, показы рекомендаций, клики, заказы, каталог товаров, продавцы, подписки
Premium и обращения в поддержку.

Ниже — первичный обзор структуры каждой таблицы, на основе которого
в следующем разделе проведём аудит качества данных.

In [5]:
np.random.seed(42)

def load_data(data):
  tables = {}
  names = ["users", "experiment_assignments", "sessions", "impressions",
              "clicks", "orders", "products", "sellers",
              "premium_subscriptions", "support_tickets"]
  for name in names:
    tables[name] = pd.read_csv(f"{data}/{name}.csv")
  return tables

tables = load_data("data")
users, exp, sessions, impressions, clicks, orders, products, sellers, premium, tickets = (
     tables["users"], tables["experiment_assignments"], tables["sessions"],
     tables["impressions"], tables["clicks"], tables["orders"], tables["products"],
     tables["sellers"], tables["premium_subscriptions"], tables["support_tickets"]
)

In [6]:
for key in tables.keys():
    print(tables[f"{key}"].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            50000 non-null  object
 1   registration_date  50000 non-null  object
 2   region             50000 non-null  object
 3   device             50000 non-null  object
 4   age_group          50000 non-null  object
 5   traffic_source     50000 non-null  object
 6   is_premium         50000 non-null  int64 
 7   is_bot_candidate   50000 non-null  int64 
dtypes: int64(2), object(6)
memory usage: 3.1+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50300 entries, 0 to 50299
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   assignment_id      50300 non-null  object
 1   user_id            50300 non-null  object
 2   experiment_name    50300 non-null  object
 3   experiment_group   

**Наблюдения по таблице `orders`:**
- 14 колонок, 7407 заказов,
- Пропуски в `delivery_fee_rub` (303 из 7407, ~4%) — требуют проверки, случайны
  ли они
- `return_date` заполнен только для 462 строк — ожидаемо (только возвращённые
  заказы), проверим согласованность с `is_returned`

**Наблюдения по схеме в целом:**
- `experiment_assignments` содержит 50300 записей, при 50000 пользователей в
  `users` — потенциально несколько экспериментов в одной таблице или дубли,
  нужно проверить `experiment_name`
- `experiment_group` присутствует и в `experiment_assignments`, и в `sessions` —
  можно свериться на консистентность
- В `users` есть флаг `is_bot_candidate`, в `impressions` — `is_duplicate_event`:
  оба нужно учесть при очистке перед расчётом метрик

## 1. Проверка качества данных

Проведём аудит по пяти направлениям: пропуски, дубли, корректность распределения
по группам эксперимента, аномалии, и в конце — свод найденных проблем с подходом
к их обработке.

In [14]:
def audit_table(df, name, pk=None):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(f"Строк: {len(df)}, колонок: {df.shape[1]}")

    na = df.isna().sum()
    na = na[na > 0]
    if len(na):
        na_pct = (na / len(df) * 100).round(2)
        print("\nПропуски:")
        print(pd.DataFrame({"count": na, "pct": na_pct}))
    else:
        print("\nПропусков нет")

    if pk:
        dup = df.duplicated(subset=pk).sum()
        print(f"\nДубликатов по PK ({pk}): {dup}")
    dup_full = df.duplicated().sum()
    print(f"Полных дубликатов строк: {dup_full}")
    return {"table": name, "n_rows": len(df), "n_na_cols": len(na), "pk_dups": dup if pk else None}


tables_pk = {
    "users": (users, "user_id"),
    "experiment_assignments": (exp, "assignment_id"),
    "sessions": (sessions, "session_id"),
    "impressions": (impressions, "impression_id"),
    "clicks": (clicks, "click_id"),
    "orders": (orders, "order_id"),
    "products": (products, "product_id"),
    "sellers": (sellers, "seller_id"),
    "premium_subscriptions": (premium, None),
    "support_tickets": (tickets, "ticket_id"),
}

audit_summary = []
for label, (df, pk) in tables_pk.items():
    audit_summary.append(audit_table(df, label, pk=pk))

audit_summary_df = pd.DataFrame(audit_summary)
audit_summary_df


users
Строк: 50000, колонок: 8

Пропусков нет

Дубликатов по PK (user_id): 0
Полных дублей строк: 0

experiment_assignments
Строк: 50300, колонок: 6

Пропусков нет

Дубликатов по PK (assignment_id): 0
Полных дублей строк: 0

sessions
Строк: 164739, колонок: 9

Пропусков нет

Дубликатов по PK (session_id): 0
Полных дублей строк: 0

impressions
Строк: 1066434, колонок: 9

Пропусков нет

Дубликатов по PK (impression_id): 0
Полных дублей строк: 0

clicks
Строк: 91266, колонок: 7

Пропусков нет

Дубликатов по PK (click_id): 0
Полных дублей строк: 0

orders
Строк: 7407, колонок: 14

Пропуски:
                  count    pct
delivery_fee_rub    303   4.09
return_date        6945  93.76

Дубликатов по PK (order_id): 0
Полных дублей строк: 0

products
Строк: 4500, колонок: 8

Пропусков нет

Дубликатов по PK (product_id): 0
Полных дублей строк: 0

sellers
Строк: 700, колонок: 5

Пропусков нет

Дубликатов по PK (seller_id): 0
Полных дублей строк: 0

premium_subscriptions
Строк: 9127, колонок: 6



,table,n_rows,n_na_cols,pk_dups
0,users,50000,0,0.0
1,experiment_assignments,50300,0,0.0
2,sessions,164739,0,0.0
3,impressions,1066434,0,0.0
4,clicks,91266,0,0.0
5,orders,7407,2,0.0
6,products,4500,0,0.0
7,sellers,700,0,0.0
8,premium_subscriptions,9127,0,NaN
9,support_tickets,337,0,0.0


### 1.1 Корректность распределения пользователей между группами

Проверяем: сколько экспериментов в данных, нет ли контаминации (юзер в обеих
группах), SRM (Sample Ratio Mismatch), баланс ковариат между control/test.

In [15]:
print(exp["experiment_name"].value_counts())
print()
print(exp["experiment_group"].value_counts())
print()
print(exp["assignment_source"].value_counts())

experiment_name
ai_recommendations_homepage    50300
Name: count, dtype: int64

experiment_group
test       25326
control    24974
Name: count, dtype: int64

assignment_source
assignment_service        50000
assignment_service_bug      300
Name: count, dtype: int64


In [17]:
buggy_users = exp[exp["assignment_source"] == "assignment_service_bug"]["user_id"].unique()
print(f"Пользователей, затронутых багом assignment_service_bug: {len(buggy_users)}")
check = exp[exp["user_id"].isin(buggy_users)].groupby("user_id")["experiment_group"].nunique()
print(f"Из них реально в разных группах A/B тестирования: {(check > 1).sum()} из {len(buggy_users)}")

Пользователей, затронутых багом assignment_service_bug: 300
Из них реально в разных группах A/B тестирования: 300 из 300


**Найдена критическая проблема: contamination групп эксперимента.**

Часть пользователей получила назначение в ОБЕ группы (`control` и `test`)
из-за бага сервиса присвоения (поле `assignment_source = "assignment_service_bug"`,
задокументировано в самих данных). Эти пользователи видели обе версии
рекомендательной системы, что нарушает предпосылку независимости групп.

**Решение:** исключить всех затронутых пользователей из дальнейшего анализа —
их поведение нельзя однозначно приписать ни одной из групп.

In [19]:
exp_clean = exp[~exp["user_id"].isin(buggy_users)].copy()

print(f"Было: {len(exp)} записей, {exp['user_id'].nunique()} юзеров")
print(f"Стало: {len(exp_clean)} записей, {exp_clean['user_id'].nunique()} юзеров")
print(f"Дублей user_id после чистки: {exp_clean['user_id'].duplicated().sum()}")

Было: 50300 записей, 50000 юзеров
Стало: 49700 записей, 49700 юзеров
Дублей user_id после чистки: 0


### SRM-тест на очищенных данных
Проведём SRM-тест на очищенных данных, чтобы определить есть ли перекос по сплиту
пользователей между control и test группами

In [20]:
def check_srm(exp_assignments, group_col="experiment_group", expected_ratio=0.5, alpha=0.01):
    counts = exp_assignments[group_col].value_counts()
    n_total = counts.sum()
    observed = counts.values
    expected = np.array([n_total * expected_ratio, n_total * (1 - expected_ratio)])
    chi2, p_value = stats.chisquare(observed, expected)
    print("--- SRM Check ---")
    print(counts)
    print(f"chi2 = {chi2:.4f}, p-value = {p_value:.6f}")
    if p_value < alpha:
        print("SRM Обнаружен")
    else:
        print("SRM не обнаружен")
    return p_value

check_srm(exp_clean)

--- SRM Check ---
experiment_group
test       25026
control    24674
Name: count, dtype: int64
chi2 = 2.4930, p-value = 0.114351
SRM не обнаружен


np.float64(0.1143507857167572)

In [21]:
merged_check = sessions.merge(
    exp_clean[["user_id", "experiment_group"]], on="user_id",
    suffixes=("_session", "_assignment"), how="inner"
)
group_mismatch = merged_check[
    merged_check["experiment_group_session"] != merged_check["experiment_group_assignment"]
]
print(f"Несовпадений experiment_group между sessions и exp_clean: "
      f"{len(group_mismatch)} из {len(merged_check)} сессий")

Несовпадений experiment_group между sessions и exp_clean: 0 из 163718 сессий


**SRM-тест пройден.** После исключения 300 пользователей, затронутых багом
`assignment_service_bug`, распределение по группам составляет test: 25026,
control: 24674 (p-value = 0.114), что статистически не отличается от ожидаемого
сплита 50/50. Рандомизация корректная, дальнейшее сравнение метрик
между группами методологически обосновано.

In [22]:
def check_covariate_balance(users_df, exp_df, covariates, user_col="user_id", group_col="experiment_group"):
    merged = exp_df.merge(users_df, on=user_col, how="left")
    for cov in covariates:
        print(f"\n--- Баланс по '{cov}' ---")
        if merged[cov].dtype == "object" or merged[cov].nunique() < 15:
            tab = pd.crosstab(merged[cov], merged[group_col], normalize="columns") * 100
            print(tab.round(2))
            ct = pd.crosstab(merged[cov], merged[group_col])
            chi2, p, _, _ = stats.chi2_contingency(ct)
            print(f"chi2 p-value = {p:.4f}", "⚠️ разбаланс" if p < 0.05 else "✅ ок")
        else:
            g1 = merged[merged[group_col] == "control"][cov].dropna()
            g2 = merged[merged[group_col] == "test"][cov].dropna()
            print(f"control: mean={g1.mean():.2f}, std={g1.std():.2f}")
            print(f"test:    mean={g2.mean():.2f}, std={g2.std():.2f}")
            stat, p = stats.mannwhitneyu(g1, g2)
            print(f"Mann-Whitney p-value = {p:.4f}", "⚠️ разбаланс" if p < 0.05 else "✅ ок")

check_covariate_balance(
    users, exp_clean,
    covariates=["device", "region", "traffic_source", "age_group", "is_premium", "is_bot_candidate"]
)


--- Баланс по 'device' ---
experiment_group  control   test
device                          
android             57.38  58.22
ios                 27.29  26.63
web                 15.34  15.15
chi2 p-value = 0.1518 ✅ ок

--- Баланс по 'region' ---
experiment_group  control   test
region                          
Central Russia      21.97  21.86
Far East             2.90   2.93
Moscow              17.95  17.93
Saint Petersburg     8.66   9.22
Siberia             12.10  11.88
South                9.20   9.14
Ural                10.18  10.14
Volga               17.03  16.90
chi2 p-value = 0.6367 ✅ ок

--- Баланс по 'traffic_source' ---
experiment_group  control   test
traffic_source                  
affiliate            6.93   7.06
direct              11.50  12.01
email                8.17   7.72
organic             42.39  42.20
paid_ads            19.25  19.01
push                11.75  12.00
chi2 p-value = 0.1983 ✅ ок

--- Баланс по 'age_group' ---
experiment_group  control   test
age_